# Model Training and Validation

**EXSU 500 — Prostate Cancer Stage Classification**

This notebook trains and validates two supervised learning algorithms:
- Logistic Regression
- Random Forest

Three genomic feature sets are evaluated:
- Somatic mutations
- Copy-number alterations (CNAs)
- Combined mutations and CNAs

All models use the same predefined patient-level training and validation splits. The held-out test set is reserved for final evaluation.

## 1. Data Loading

Load the three genomic feature matrices and patient-level split assignments generated in `02_split_preprocessing.ipynb`.

The dataset contains 1,967 patients:
- Mutation-only: 341 features
- CNA-only: 340 features
- Combined: 681 features

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path("../data/processed")
RANDOM_STATE = 42

# Load the three feature sets prepared in notebook 02
feature_files = {
    "Mutation": "mutation_feature_matrix.csv",
    "CNA": "cna_feature_matrix_common.csv",
    "Combined": "combined_feature_matrix.csv",
}

feature_sets = {
    name: pd.read_csv(DATA_DIR / filename, index_col="PATIENT_ID")
    for name, filename in feature_files.items()
}

splits = pd.read_csv(DATA_DIR / "patient_splits.csv")

print("Feature matrix dimensions:")
for name, X in feature_sets.items():
    print(f"  {name}: {X.shape}")

print("\nPatients per split:")
print(splits["split"].value_counts())

## 2. Data Validation

Verify that genomic features and patient labels are correctly aligned.

Checks include:
- Unique patient identifiers
- Consistent patients across all feature sets
- Valid binary stage labels
- No missing or non-finite feature values
- No overlap between training and validation patients

The predefined 70/15/15 split is preserved.

In [ ]:
required_columns = {"PATIENT_ID", "stage_binary", "split"}

assert required_columns.issubset(splits.columns)
assert splits["PATIENT_ID"].is_unique
assert splits["stage_binary"].isin([0, 1]).all()
assert set(splits["split"]) == {"train", "validation", "test"}

train_ids = splits.loc[
    splits["split"] == "train", "PATIENT_ID"
].tolist()

val_ids = splits.loc[
    splits["split"] == "validation", "PATIENT_ID"
].tolist()

assert set(train_ids).isdisjoint(val_ids)

y_by_patient = splits.set_index("PATIENT_ID")["stage_binary"]

y_train = y_by_patient.loc[train_ids]
y_val = y_by_patient.loc[val_ids]

for name, X in feature_sets.items():
    assert X.index.is_unique
    assert X.columns.is_unique
    assert set(X.index) == set(y_by_patient.index)
    assert np.isfinite(X.to_numpy(dtype=float)).all()

assert (
    feature_sets["Mutation"].shape[1]
    + feature_sets["CNA"].shape[1]
    == feature_sets["Combined"].shape[1]
)

print("PASS: Feature sets and patient splits validated.")
print(f"Training patients: {len(train_ids)}")
print(f"Validation patients: {len(val_ids)}")

Stage 1–3: 1000
Stage 4: 1000


## 3. Model Configuration

Two classification algorithms are evaluated:

**Logistic Regression**
- Linear classification baseline
- Standardization of genomic features
- Regularization to reduce overfitting

**Random Forest**
- Nonlinear ensemble classifier
- Captures complex relationships between genomic features
- Does not require feature scaling

Constant features are removed using `VarianceThreshold`.

All preprocessing steps are fitted using training data only to prevent information leakage.

In [ ]:
def build_model(model_name):
    steps = [
        ("nonconstant", VarianceThreshold(threshold=0.0))
    ]

    if model_name == "Logistic Regression":
        steps += [
            ("scale", StandardScaler()),
            ("classifier", LogisticRegression(
                C=1.0,
                max_iter=3000,
                solver="liblinear",
                random_state=RANDOM_STATE,
            )),
        ]

    elif model_name == "Random Forest":
        steps += [
            ("classifier", RandomForestClassifier(
                n_estimators=300,
                min_samples_leaf=2,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ]

    else:
        raise ValueError(f"Unknown model: {model_name}")

    return Pipeline(steps)


model_names = [
    "Logistic Regression",
    "Random Forest"
]

Mutations: (2000, 50)
CNA: (2000, 50)
Combined: (2000, 100)


## 4. Model Training and Validation

Train both algorithms on each genomic feature set, producing six model configurations.

Models are evaluated on the validation set using:
- AUROC: Discrimination between Stage I–III and Stage IV
- Average Precision (AP): Summary of precision-recall performance

Results are compared across all six configurations.

Validation metrics and prediction probabilities are saved for subsequent analysis. The held-out test set is not used during this stage.

In [ ]:
results = []
trained_models = {}

validation_predictions = pd.DataFrame({
    "PATIENT_ID": val_ids,
    "stage_binary": y_val.to_numpy(),
})

for feature_name, X in feature_sets.items():
    X_train = X.loc[train_ids]
    X_val = X.loc[val_ids]

    for model_name in model_names:
        pipeline = build_model(model_name)
        pipeline.fit(X_train, y_train)

        val_prob = pipeline.predict_proba(X_val)[:, 1]

        auroc = roc_auc_score(y_val, val_prob)
        ap = average_precision_score(y_val, val_prob)

        trained_models[(feature_name, model_name)] = pipeline

        results.append({
            "Feature Set": feature_name,
            "Model": model_name,
            "Features Used": int(
                pipeline.named_steps["nonconstant"]
                .get_support().sum()
            ),
            "Validation AUROC": auroc,
            "Validation AP": ap,
        })

        validation_predictions[
            f"{feature_name}__{model_name}"
        ] = val_prob

        print(
            f"{feature_name:8} | {model_name:20} "
            f"| AUROC={auroc:.3f} | AP={ap:.3f}"
        )

results_df = pd.DataFrame(results).sort_values(
    ["Validation AUROC", "Validation AP"],
    ascending=False
).reset_index(drop=True)

results_df.to_csv(
    DATA_DIR / "validation_model_comparison.csv",
    index=False
)

validation_predictions.to_csv(
    DATA_DIR / "validation_predictions.csv",
    index=False
)

print("\nValidation comparison:")
print(results_df.to_string(
    index=False,
    float_format=lambda x: f"{x:.3f}"
))

print("\nValidation results saved.")

Train: 1400
Validation: 300
Test: 300
